# 06 — IFRS 9 Concepts: Staging & Expected Credit Loss (ECL)

## Objective
Create an educational implementation of stage assignment and scenario-weighted ECL arithmetic.

**Critical limitation:** the UCI dataset only has a next-month default outcome. It does not provide contractual maturity, lifetime default curves, effective interest rates, forward-looking macroeconomic scenarios, or institution-specific SICR policy. Therefore, this notebook is a conceptual demonstration and is NOT an IFRS 9-compliant impairment model.

In [1]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 1. IFRS 9 staging concepts

- **Stage 1:** 12-month ECL when credit risk has not increased significantly since initial recognition.
- **Stage 2:** lifetime ECL following a significant increase in credit risk (SICR).
- **Stage 3:** lifetime ECL for credit-impaired assets.

The 30-days-past-due presumption and 90-days-past-due rebuttable presumption are important concepts, but actual staging requires documented policy, rebuttal assessment, qualitative indicators, and forward-looking information.

In [2]:
from src.ifrs9 import assign_stage, calculate_ecl
stage_examples = pd.DataFrame({
    "days_past_due": [0, 15, 35, 95, 0],
    "significant_increase": [False, False, False, False, True],
    "credit_impaired": [False, False, False, False, False]
})
stage_examples["stage"] = stage_examples.apply(
    lambda r: assign_stage(r.days_past_due, r.significant_increase, r.credit_impaired),
    axis=1
)
display(stage_examples)

,days_past_due,significant_increase,credit_impaired,stage
0,0,False,False,1
1,15,False,False,1
2,35,False,False,2
3,95,False,False,3
4,0,True,False,2


## 2. Scenario-weighted ECL example

For this simplified example, ECL is calculated as PD × LGD × EAD × discount factor, weighted across macroeconomic scenarios. A real IFRS 9 calculation uses discounted expected cash shortfalls over the relevant horizon and scenario-consistent lifetime risk parameters.

In [3]:
ecl, scenario_detail = calculate_ecl(
    pd_value=0.08,
    lgd=0.45,
    ead=100000,
    discount_factor=0.97,
    scenario_weights={"downside": 0.20, "base": 0.60, "upside": 0.20},
    scenario_pd_multipliers={"downside": 1.50, "base": 1.00, "upside": 0.75}
)
print(f"Illustrative weighted ECL: {ecl:,.2f}")
display(scenario_detail)

Illustrative weighted ECL: 3,666.60


,scenario,weight,scenario_pd,lgd,ead,discount_factor,scenario_ecl,weighted_ecl
0,downside,0.2,0.12,0.45,100000,0.97,5238.0,1047.6
1,base,0.6,0.08,0.45,100000,0.97,3492.0,2095.2
2,upside,0.2,0.06,0.45,100000,0.97,2619.0,523.8


## 3. Portfolio-level illustrative ECL

In [4]:
from src.ifrs9 import calculate_portfolio_ecl
portfolio = pd.DataFrame({
    "account_id": ["A001", "A002", "A003", "A004"],
    "PD": [0.02, 0.08, 0.35, 0.60],
    "LGD": [0.40, 0.45, 0.50, 0.60],
    "EAD": [50000, 100000, 75000, 30000],
    "STAGE": [1, 1, 2, 3]
})
portfolio_ecl = calculate_portfolio_ecl(portfolio)
display(portfolio_ecl)
print("Total illustrative ECL:", round(portfolio_ecl["ECL"].sum(), 2))

,account_id,PD,LGD,EAD,STAGE,ECL
0,A001,0.02,0.40,50000,1,400.0
1,A002,0.08,0.45,100000,1,3600.0
2,A003,0.35,0.50,75000,2,13125.0
3,A004,0.60,0.60,30000,3,10800.0


Total illustrative ECL: 27925.0


## 4. Production requirements and governance

A production IFRS 9 framework additionally requires:
1. Initial-recognition and reporting-date credit risk comparison for SICR.
2. Lifetime PD term structures and appropriate 12-month PD.
3. EAD profiles including expected drawdowns and repayments.
4. LGD estimates incorporating recoveries, costs, collateral and timing.
5. Effective-interest-rate discounting.
6. Multiple probability-weighted, supportable forward-looking scenarios.
7. Documented staging policy, model validation, overlays, controls and audit trail.

The simple stage function and arithmetic in this notebook are teaching aids only. Do not use them for financial statements or lending decisions.